# 02 · Clean data — coercion, missing-value classification, outlier register

**No imputation, no row dropping, no window restriction** (that is notebook `03`).
What happens here:

1. Coerce each cell to a float, trusting the *native* numeric value from the
   workbook and only string-parsing genuinely non-numeric cells.
2. Classify every absent value into a `missing_reason`
   (`hostel_room_occ_not_published`, `covid_hole`, `covid_recovery_gap`,
   `provisional_gap`, `series_not_started`, `na_token`, …).
3. Reshape into one long monthly panel `date × hotel_category × variable`,
   a quarterly capacity table, and a monthly CPI table with the two deflators.
4. Build a **flag-only** outlier register (nothing is removed).

**Inputs** `data/processed/01_loaded/*.parquet` →
**Outputs** `02_panel_long.parquet`, `02_capacity_quarterly.parquet`,
`02_cpi_monthly.parquet`, plus `outputs/tables/missing_value_map.csv` and
`outputs/tables/outlier_register.csv`.

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd

SRC = Path.cwd().parent / "src"
sys.path.insert(0, str(SRC))
import config as C
from common import AuditLog, is_na_token, to_number

LOG = AuditLog("02_clean")
L = C.P_LOADED_DIR
VARLABEL = {"room_occupancy": "room_occupancy", "bed_occupancy": "bed_occupancy",
            "average_rate": "average_rate", "travelers": "travelers"}

## Numeric coercion + missing-value reason

`classify_missing` runs only on rows whose numeric value is `NaN`, so a
non-empty `missing_reason` is by construction mutually exclusive with a present
value. COVID windows are the dates locked in `config.py` (`2020-03…2021-12`
hard hole, `2022` degraded recovery).

In [2]:
def classify_missing(row):
    if not np.isnan(row["value"]):
        return ""
    d, var, cat = row["date"], row["variable"], row["hotel_category"]
    tok = str(row["raw_value"]).strip().lower()
    if var == "room_occupancy" and cat == "hostel" and tok in {"---", "--", "-"}:
        return "hostel_room_occ_not_published"
    if var == "travelers" and cat in {"hostel", "otros_resto", "total_parahoteleros"}:
        return "not_in_series_scope"
    if var == "travelers" and d < pd.Timestamp(2013, 1, 1):
        return "series_not_started"
    if C.COVID_HOLE_START <= d.strftime("%Y-%m-%d") <= C.COVID_HOLE_END:
        return "covid_hole"
    if C.COVID_RECOVERY_START <= d.strftime("%Y-%m-%d") <= C.COVID_RECOVERY_END:
        return "covid_recovery_gap"
    if d >= pd.Timestamp(2025, 12, 1):
        return "provisional_gap"
    if is_na_token(row["raw_value"]):
        return "na_token"
    return "unparseable"


def load_and_coerce(name):
    df = pd.read_parquet(L / f"{name}.parquet").copy()
    df["variable"] = VARLABEL[name]
    parsed = df["raw_value"].where(~df["cell_is_numeric"]).map(
        lambda v: to_number(v) if isinstance(v, str) else (float("nan"), False, False))
    df["value"] = np.where(df["cell_is_numeric"], df["cell_value"],
                           [p[0] for p in parsed])
    df["_unparseable"] = [p[2] for p in parsed]
    if int(df["_unparseable"].sum()):
        ex = df.loc[df["_unparseable"], "raw_value"].value_counts().head(5).to_dict()
        LOG.log("unparseable_cells", name, int(df["_unparseable"].sum()), str(ex))
    df["missing_reason"] = df.apply(classify_missing, axis=1)
    LOG.log("coerced", name, len(df),
            f"{df['value'].notna().sum()} numeric / {df['value'].isna().sum()} NaN")
    return df[["date", "hotel_category", "variable", "value", "raw_value",
               "missing_reason", "provisional", "src_sheet", "src_row", "src_col"]]

In [3]:
parts = [load_and_coerce(n) for n in
         ["average_rate", "room_occupancy", "bed_occupancy", "travelers"]]
panel = pd.concat(parts, ignore_index=True).sort_values(
    ["variable", "hotel_category", "date"]).reset_index(drop=True)

bad_occ = panel[(panel.variable.str.contains("occupancy")) & panel.value.notna() &
                ((panel.value < 0) | (panel.value > 100))]
bad_rate = panel[(panel.variable == "average_rate") & panel.value.notna() &
                 (panel.value <= 0)]
LOG.log("range_check_occupancy", "0..100", len(bad_occ),
        "ok" if not len(bad_occ) else "OUT OF RANGE")
LOG.log("range_check_rate", ">0", len(bad_rate),
        "ok" if not len(bad_rate) else "NON-POSITIVE")
assert len(bad_occ) == 0 and len(bad_rate) == 0, "range check failed"

mm = (panel.groupby(["variable", "missing_reason"]).size().rename("n")
      .reset_index().sort_values(["variable", "missing_reason"]))
mm.to_csv(C.OUT_TAB / "missing_value_map.csv", index=False)
mm

  [02_clean] coerced                average_rate                         1768  1540 numeric / 228 NaN
  [02_clean] coerced                room_occupancy                       2210  1794 numeric / 416 NaN
  [02_clean] coerced                bed_occupancy                        2210  1984 numeric / 226 NaN
  [02_clean] coerced                travelers                             966  822 numeric / 144 NaN
  [02_clean] range_check_occupancy  0..100                                  0  ok
  [02_clean] range_check_rate       >0                                      0  ok


,variable,missing_reason,n
0,average_rate,,1540
1,average_rate,covid_hole,176
2,average_rate,covid_recovery_gap,36
3,average_rate,provisional_gap,16
4,bed_occupancy,,1984
5,bed_occupancy,covid_hole,190
6,bed_occupancy,covid_recovery_gap,36
7,room_occupancy,,1794
8,room_occupancy,covid_hole,170
9,room_occupancy,covid_recovery_gap,36


## Capacity → quarterly, wide by metric

In [4]:
cap = pd.read_parquet(L / "capacity.parquet").copy()
cap["value"] = np.where(cap["cell_is_numeric"], cap["cell_value"],
                        cap["raw_value"].map(lambda v: to_number(v)[0]))
cap_w = (cap.pivot_table(index=["date", "snapshot_month", "hotel_category",
                                "source_taxonomy"],
                         columns="metric", values="value", aggfunc="first")
         .reset_index())
cap_w.columns.name = None
for _, r in cap_w[(cap_w.establishments.notna()) &
                  (cap_w.establishments % 1 != 0)].iterrows():
    LOG.log("noninteger_establishments", f"{r['date']:%Y-%m} {r['hotel_category']}",
            round(r["establishments"], 2), "kept, flagged (source-imputed)")
cap_w["is_provisional_2023plus"] = cap_w["date"].dt.year >= 2023
cap_w = cap_w.sort_values(["hotel_category", "date"]).reset_index(drop=True)
LOG.log("capacity_wide", "02_capacity_quarterly", len(cap_w),
        f"{cap_w.date.min():%Y-%m}..{cap_w.date.max():%Y-%m}")
cap_w.tail(4)

  [02_clean] noninteger_establishments 2020-12 total_hoteleros            169.81  kept, flagged (source-imputed)
  [02_clean] capacity_wide          02_capacity_quarterly                 530  2008-03..2026-03


,date,snapshot_month,hotel_category,source_taxonomy,available_bed_nights,available_room_nights,establishments,is_provisional_2023plus
526,2025-06-01,6,total_parahoteleros,ehoba,156660.0,60270.0,61.0,True
527,2025-09-01,9,total_parahoteleros,ehoba,156480.0,60540.0,61.0,True
528,2025-12-01,12,total_parahoteleros,ehoba,166098.0,63736.0,61.0,True
529,2026-03-01,3,total_parahoteleros,ehoba,152954.0,58683.0,68.0,True


## CPI monthly — build the two deflators + INDEC MoM cross-check

* `cpi_gba` = GBA bridge (2016-04…2016-11) spliced onto the GBA block of the
  index sheet (2016-12…). Both are already on the dic-2016 = 100 base, so this
  is a level continuation — verified by the overlap check at 2016-12.
* `cpi_nac` = Total-nacional `Nivel general` (2016-12…).
* `cpi_resthot_*` = the `Restaurantes y hoteles` COICOP division (context only —
  a consumer-price index, **not** a hotel cost index).
* Cross-check: MoM recomputed from `cpi_gba` levels vs INDEC's published MoM.

In [5]:
cpi = pd.read_parquet(L / "cpi.parquet").copy()
cpi["s"] = cpi["series"].str.strip().str.lower()


def pick(sheet, region, s):
    m = cpi[(cpi.src_sheet == sheet) & (cpi.region == region) & (cpi.s == s)]
    return m.set_index("date")["value"].sort_index()


idx_gba = pick("index_national", "gba", "nivel general")
bridge = pick("gba_bridge", "gba", "nivel general")
idx_nac = pick("index_national", "total_nacional", "nivel general")
mom_gba_pub = pick("mom_national", "gba", "nivel general")
mom_nac_pub = pick("mom_national", "total_nacional", "nivel general")

ov = bridge.index.intersection(idx_gba.index)
LOG.log("cpi_splice_check", "gba bridge vs index @ overlap",
        f"{(bridge[ov] - idx_gba[ov]).abs().max():.4f}",
        "max abs index-point diff (expect ~0 at 2016-12)")

cpi_gba = pd.concat([bridge[bridge.index < idx_gba.index.min()], idx_gba]).sort_index()
out = pd.DataFrame({"cpi_gba": cpi_gba})
out["cpi_nac"] = idx_nac
out["cpi_resthot_gba"] = pick("index_national", "gba", "restaurantes y hoteles")
out["cpi_resthot_nac"] = pick("index_national", "total_nacional", "restaurantes y hoteles")
out["indec_mom_gba_pct"] = mom_gba_pub * (100 if mom_gba_pub.abs().median() < 1 else 1)
out["indec_mom_nac_pct"] = mom_nac_pub * (100 if mom_nac_pub.abs().median() < 1 else 1)
out = out.sort_index()
out.index.name = "date"
out = out.reset_index()

chk = out.assign(mom_calc_gba=out["cpi_gba"].pct_change() * 100).dropna(
    subset=["mom_calc_gba", "indec_mom_gba_pct"])
gap = (chk["mom_calc_gba"] - chk["indec_mom_gba_pct"]).abs()
LOG.log("cpi_mom_crosscheck", "calc vs INDEC (GBA)",
        f"max={gap.max():.3f}pp", f"mean={gap.mean():.4f}pp over {len(chk)} months")
LOG.log("cpi_monthly", "02_cpi_monthly", len(out),
        f"{out.date.min():%Y-%m}..{out.date.max():%Y-%m}; "
        f"cpi_nac starts {out.dropna(subset=['cpi_nac']).date.min():%Y-%m}")
out.tail(4)

  [02_clean] cpi_splice_check       gba bridge vs index @ overlap      0.0000  max abs index-point diff (expect ~0 at 2016-12)
  [02_clean] cpi_mom_crosscheck     calc vs INDEC (GBA)              max=0.050pp  mean=0.0245pp over 114 months
  [02_clean] cpi_monthly            02_cpi_monthly                        124  2016-04..2026-07; cpi_nac starts 2016-12


,date,cpi_gba,cpi_nac,cpi_resthot_gba,cpi_resthot_nac,indec_mom_gba_pct,indec_mom_nac_pct
120,2026-04-01,11336.5576,11363.0904,13841.4078,14041.1974,NaN,NaN
121,2026-05-01,11594.5499,11607.3937,14129.9031,14291.4848,2.3,2.1
122,2026-06-01,11810.9464,11826.4103,14294.4029,14524.0788,1.9,1.9
123,2026-07-01,12078.7372,12076.3937,14759.1647,14923.8212,2.3,2.1


## Outlier register (flag only — nothing removed)

Rule: a monthly log-change in `average_rate` bigger than 0.40 that is reversed
the following month (opposite sign, magnitude > 0.25). Captures one-month data
spikes without touching genuine step-changes such as the Dec-2023 devaluation.

In [6]:
recs = []
sub = panel[(panel.variable == "average_rate") & panel.value.notna()]
for cat, g in sub.groupby("hotel_category"):
    g = g.sort_values("date")
    dln = np.log(g.value).diff()
    nxt = dln.shift(-1)
    for d, v, dl, rv in zip(g.date, g.value, dln, nxt):
        if np.isfinite(dl) and abs(dl) > 0.40 and np.isfinite(rv) and \
                np.sign(rv) == -np.sign(dl) and abs(rv) > 0.25:
            recs.append({"variable": "average_rate", "hotel_category": cat,
                         "date": d, "value": round(v, 2),
                         "dln": round(float(dl), 3), "dln_next": round(float(rv), 3),
                         "note": "one-month spike then reversion"})
outliers = pd.DataFrame(recs)
outliers.to_csv(C.OUT_TAB / "outlier_register.csv", index=False)
LOG.log("outlier_register", "outlier_register.csv", len(outliers), "flagged, NOT removed")
outliers

  [02_clean] outlier_register       outlier_register.csv                    1  flagged, NOT removed


,variable,hotel_category,date,value,dln,dln_next,note
0,average_rate,stars_5,2025-08-01,381800.96,0.529,-0.355,one-month spike then reversion


## Write outputs

In [7]:
panel.to_parquet(C.P_PANEL_LONG)
cap_w.to_parquet(C.P_CAPACITY_Q)
out.to_parquet(C.P_CPI_MONTHLY)
LOG.log("write", "data/processed",
        detail="02_panel_long / 02_capacity_quarterly / 02_cpi_monthly")
LOG.flush()
print("02_clean_data complete.")

  [02_clean] write                  data/processed                             02_panel_long / 02_capacity_quarterly / 02_cpi_monthly
02_clean_data complete.
